# Train wakeword "vừng ơi" trên Colab

Notebook này thay cho việc train dưới WSL — máy đó chỉ có 8 GB RAM và đã bị OOM hai lần
tại vòng đánh giá đầu tiên (bước 500).

**Bạn cần chuẩn bị:** file `mau-vung-oi.tar.gz` (32 MB, gồm 190 mẫu giọng thật + 1000 mẫu TTS).

**Chạy theo thứ tự từ trên xuống.** Ô nào cũng in ra thứ để tự kiểm, đừng bỏ qua phần in ra.

**Runtime:** `Thời gian chạy → Thay đổi loại thời gian chạy → T4 GPU`
(giao diện tiếng Anh: `Runtime → Change runtime type`).
Góc dưới phải hiện `Đang kết nối với T4` là đã đúng, không phải chỉnh gì.
Không có GPU vẫn chạy được, chỉ chậm hơn nhiều.

⚠️ **Đừng bấm "Chạy tất cả" ngay.** Chạy từng ô cho 5 ô đầu — ô nào lỗi mà chạy tiếp thì
lỗi dây chuyền, khó tìm gốc. Ô 6 sẽ hiện hộp chọn file, phải ngồi đó lúc đó.

---

## 1. Kiểm môi trường

In [ ]:
# In ra để BIẾT mình đang có gì, không đoán. Máy WSL chết vì đúng chỗ này không được kiểm.
import subprocess, shutil

gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
print("GPU :", gpu if gpu else "KHÔNG có (vẫn train được, chỉ chậm hơn nhiều)")

ram_kb = int(open("/proc/meminfo").readline().split()[1])
print(f"RAM : {ram_kb/1024/1024:.1f} GB")

t, u, f = shutil.disk_usage("/content")
print(f"Đĩa : {f/1e9:.0f} GB trống / {t/1e9:.0f} GB")
print("\nCần tối thiểu: RAM ≥ 12 GB, đĩa trống ≥ 40 GB.")

## 2. Cài microWakeWord

In [ ]:
# Thứ tự QUAN TRỌNG: cài microwakeword trước, ép `datasets<4` sau.
#   - datasets 5.x đòi `torchcodec` để giải mã audio -> kéo cả torch vào, và đã làm hỏng
#     hai lần cài trước đó. Bản 3.x dùng soundfile, đủ dùng.
#   - tensorboard KHÔNG đi kèm TF 2.21 nữa, nhưng microwakeword gọi tf.summary.scalar ở mỗi
#     lần eval -> thiếu nó thì train chết ở đúng bước 500. Đây là lỗi thật đã gặp.
import subprocess, sys, importlib, os

# KHÔNG dùng `| tail -n` ở đây: nó nuốt mất thông báo lỗi, khiến lệnh cài hỏng mà vẫn
# trông như xong — rồi mãi tới ô "Sinh đặc trưng" mới lòi ra ModuleNotFoundError.
!git clone -q https://github.com/kahrendt/microWakeWord.git /content/mww-src

r = subprocess.run([sys.executable, "-m", "pip", "install", "-e", "/content/mww-src"],
                   capture_output=True, text=True)
if r.returncode != 0:
    print("--- pip stdout ---\n", r.stdout[-1500:])
    print("--- pip stderr ---\n", r.stderr[-1500:])
    raise SystemExit("!! cài microwakeword THẤT BẠI — đọc stderr bên trên")
print("cài microwakeword: xong")

# Nạp thẳng từ thư mục clone làm đường lui, phòng khi bản editable không vào sys.path.
if "/content/mww-src" not in sys.path:
    sys.path.insert(0, "/content/mww-src")
importlib.invalidate_caches()

r2 = subprocess.run([sys.executable, "-m", "pip", "install", "datasets<4", "tensorboard"],
                    capture_output=True, text=True)
if r2.returncode != 0:
    print(r2.stderr[-1500:])
    raise SystemExit("!! cài datasets/tensorboard THẤT BẠI")
print("cài datasets<4 + tensorboard: xong")

# CHỨNG MINH nạp được, đừng tin là nó tự chạy.
import microwakeword, tensorflow as tf
print("\nmicrowakeword :", microwakeword.__file__)
print("TF            :", tf.__version__)
print("GPU thấy được :", tf.config.list_physical_devices("GPU"))

## 3. Vá `data.py` — bỏ bản sao mảng thừa

Đây là lỗi đã giết vòng train dưới WSL, **không phải** tuỳ chọn tối ưu.

`get_data()` cắt tập ambient thành cửa sổ chồng lấn (bước 30 khung) → với chime6 9,7 giờ ra
116 045 cửa sổ 204×40 float32 = **3,8 GB**. Rồi dòng cuối trả `data[indices]` — fancy indexing
**luôn cấp phát bản sao đầy đủ** → **7,6 GB**.

Bản sao đó vô nghĩa: `model.evaluate` chỉ trả loss/tp/fp/fn/auc, toàn số tổng hợp không phụ
thuộc thứ tự. Colab có 12,7 GB nên *có thể* sống sót, nhưng vá thì nhanh hơn và chắc chắn hơn.

In [ ]:
from pathlib import Path

p = Path("/content/mww-src/microwakeword/data.py")
s = p.read_text()

CU = """        indices = np.arange(labels.shape[0])"""
MOI = """        # Tập ambient chỉ đi vào model.evaluate, trả loss/tp/fp/fn/auc — toàn số tổng hợp
        # KHÔNG phụ thuộc thứ tự. Nhưng `data[indices]` dưới kia là fancy indexing, LUÔN cấp
        # phát bản sao đầy đủ: 3,8 GB thành 7,6 GB và bị OOM-kill.
        if mode.endswith("_ambient"):
            return data, labels, weights

        indices = np.arange(labels.shape[0])"""

if "endswith(\"_ambient\")" in s:
    print("data.py: đã vá rồi")
elif CU in s:
    p.write_text(s.replace(CU, MOI, 1))
    print("data.py: VÁ XONG")
else:
    raise SystemExit("!! Không tìm thấy chỗ vá — upstream đã đổi. Mở data.py xem hàm get_data.")


# --- Vá thứ hai: train.py, chia lát vòng đánh giá ambient ---
#
# Vá data.py mới bỏ được bản sao numpy. Nhưng `model.evaluate` nhận mảng numpy rồi
# TensorFlow COPY toàn bộ sang tensor — lại 3,8 GB nữa. Đo 07/08 trên Colab: tiến trình bị
# giết (^C) đúng sau dòng "Allocation of 3786958080 exceeds 10% of free system memory".
#
# Chia lát giữ đỉnh ở mức mảng gốc + một lát (~0,65 GB). Kết quả KHÔNG đổi vì reset_metrics
# đã bị tắt ngay dòng trên, nên số đo cộng dồn qua từng lần gọi.
q = Path("/content/mww-src/microwakeword/train.py")
t = q.read_text()

CU_T = """        # XXX: tf no longer provides a way to evaluate a model without updating metrics
        with swap_attribute(model, "reset_metrics", lambda: None):
            ambient_predictions = model.evaluate(
                ambient_testing_fingerprints,
                ambient_testing_ground_truth,
                batch_size=1024,
                return_dict=True,
                verbose=0,
            )"""

MOI_T = """        # XXX: tf no longer provides a way to evaluate a model without updating metrics
        with swap_attribute(model, "reset_metrics", lambda: None):
            # Chia lát: đưa cả mảng 3,8 GB cho evaluate thì TF copy sang tensor -> 7,6 GB
            # và tiến trình bị giết. Số đo cộng dồn nên kết quả không đổi.
            LAT = 20000
            for i in range(0, len(ambient_testing_fingerprints), LAT):
                ambient_predictions = model.evaluate(
                    ambient_testing_fingerprints[i : i + LAT],
                    ambient_testing_ground_truth[i : i + LAT],
                    batch_size=1024,
                    return_dict=True,
                    verbose=0,
                )"""

if "LAT = 20000" in t:
    print("train.py: đã vá rồi")
elif CU_T in t:
    q.write_text(t.replace(CU_T, MOI_T, 1))
    print("train.py: VÁ XONG")
else:
    raise SystemExit("!! Không tìm thấy chỗ vá train.py — upstream đã đổi.")

## 4. Tải tập âm (5,4 GB nén)

In [ ]:
# Tập âm chính chủ của tác giả microWakeWord. Trên Colab tải rất nhanh (mạng Google).
import os, zipfile
from huggingface_hub import hf_hub_download

W = "/content/mww"
NEG = f"{W}/negative_datasets"
os.makedirs(NEG, exist_ok=True)

for z in ["speech.zip", "no_speech.zip", "dinner_party.zip", "dinner_party_eval.zip"]:
    dich = f"{NEG}/{z[:-4]}"
    if os.path.isdir(dich) and os.listdir(dich):
        print(f"  {z}: có rồi")
        continue
    print(f"  tải {z} ...", flush=True)
    p = hf_hub_download("kahrendt/microwakeword", z, repo_type="dataset")
    with zipfile.ZipFile(p) as f:
        f.extractall(NEG)
    os.remove(p)   # xoá zip ngay, đĩa Colab không dư dả
    print(f"  {z}: xong")

print("\n=== cấu trúc (phải thấy training/ và *_ambient/) ===")
for d in sorted(os.listdir(NEG)):
    print(" ", d, "->", sorted(os.listdir(f"{NEG}/{d}"))[:4])

## 5. Tải nguồn tăng cường (phản xạ phòng + nhạc + tiếng nền)

In [ ]:
# Ba nguồn này dùng để trộn vào mẫu dương lúc sinh spectrogram: mô phỏng tiếng vọng phòng,
# nhạc nền, tiếng ồn sinh hoạt. Thiếu chúng thì model chỉ nhận đúng giọng thu trong phòng yên.
import os, io, glob, zipfile, shutil, urllib.request
import numpy as np, soundfile as sf, librosa
from huggingface_hub import snapshot_download

W = "/content/mww"
os.chdir(W)
SR = 16000


def sang_wav16k(files, out_dir, gioi_han=None):
    os.makedirs(out_dir, exist_ok=True)
    n = 0
    for f in files:
        if gioi_han and n >= gioi_han:
            break
        ten = os.path.splitext(os.path.basename(f))[0] + ".wav"
        if os.path.exists(f"{out_dir}/{ten}"):
            n += 1
            continue
        try:
            y, _ = librosa.load(f, sr=SR, mono=True)
        except Exception:
            continue
        if len(y) < SR // 2:      # dưới 0,5 s thì vô dụng làm nền
            continue
        sf.write(f"{out_dir}/{ten}", (y * 32767).astype(np.int16), SR, subtype="PCM_16")
        n += 1
    print(f"  -> {out_dir}: {len(glob.glob(out_dir + '/*.wav'))} file")


# --- 1/3 phản xạ phòng (MIT RIR) — quan trọng nhất
if len(glob.glob("mit_rirs/*.wav")) < 100:
    print("[1/3] MIT RIR", flush=True)
    # snapshot_download tải SONG SONG. Tải từng file một bị HF bóp tốc độ khi không có token
    # (đo thật: ~3 file/phút => 270 file mất 90 phút).
    kho = snapshot_download("davidscripka/MIT_environmental_impulse_responses",
                            repo_type="dataset", allow_patterns=["16khz/*.wav"], max_workers=8)
    os.makedirs("mit_rirs", exist_ok=True)
    for f in glob.glob(f"{kho}/16khz/*.wav"):
        shutil.copy(f, "mit_rirs/")
    print(f"  -> mit_rirs: {len(glob.glob('mit_rirs/*.wav'))} file")
else:
    print("[1/3] MIT RIR: có rồi")

# --- 2/3 nhạc (FMA extra small)
if len(glob.glob("fma_16k/*.wav")) < 50:
    print("[2/3] FMA", flush=True)
    os.makedirs("fma", exist_ok=True)
    if not glob.glob("fma/**/*.mp3", recursive=True):
        urllib.request.urlretrieve(
            "https://huggingface.co/datasets/mchl914/fma_xsmall/resolve/main/fma_xs.zip",
            "fma/fma_xs.zip")
        with zipfile.ZipFile("fma/fma_xs.zip") as f:
            f.extractall("fma")
        os.remove("fma/fma_xs.zip")
    sang_wav16k(sorted(glob.glob("fma/**/*.mp3", recursive=True)), "fma_16k", gioi_han=250)
    shutil.rmtree("fma", ignore_errors=True)   # mp3 gốc hết việc, xoá cho nhẹ đĩa
else:
    print("[2/3] FMA: có rồi")

# --- 3/3 tiếng nền (AudioSet)
if len(glob.glob("audioset_16k/*.wav")) < 50:
    print("[3/3] AudioSet", flush=True)
    # BẪY: file .tar trong repo này trả 404 — kho đã chuyển sang parquet shard.
    import pandas as pd
    os.makedirs("audioset_16k", exist_ok=True)
    n = 0
    for i in range(3):
        p = hf_hub_download("agkphysics/AudioSet", f"data/bal_train/{i:02d}.parquet",
                            repo_type="dataset")
        df = pd.read_parquet(p)
        for row in df["audio"]:
            if n >= 2000:
                break
            try:
                y, _ = librosa.load(io.BytesIO(row["bytes"]), sr=SR, mono=True)
            except Exception:
                continue
            if len(y) < SR // 2:
                continue
            sf.write(f"audioset_16k/as_{n:05d}.wav", (y * 32767).astype(np.int16), SR,
                     subtype="PCM_16")
            n += 1
        os.remove(p)
        if n >= 2000:
            break
    print(f"  -> audioset_16k: {n} file")
else:
    print("[3/3] AudioSet: có rồi")

for d in ("mit_rirs", "fma_16k", "audioset_16k"):
    print(f"{d}: {len(glob.glob(d + '/*.wav'))} file")

## 6. Tải mẫu giọng lên

Chạy ô dưới rồi chọn file **`mau-vung-oi.tar.gz`** trên máy bạn.

Nếu file nằm sẵn trên Google Drive thì thay bằng:
```python
from google.colab import drive
drive.mount('/content/drive')
!tar xzf /content/drive/MyDrive/mau-vung-oi.tar.gz -C /content/mww
```

In [ ]:
import os, glob
os.chdir("/content/mww")

if not glob.glob("real_samples/*.wav"):
    from google.colab import files
    up = files.upload()          # chọn mau-vung-oi.tar.gz
    ten = list(up.keys())[0]
    !tar xzf "{ten}" -C /content/mww

r = len(glob.glob("real_samples/*.wav"))
g = len(glob.glob("generated_samples/*.wav"))
print(f"mẫu THẬT (giọng bạn) : {r}")
print(f"mẫu TTS (tổng hợp)   : {g}")
assert r > 100, "!! thiếu mẫu thật — đây là thứ MANG THANH ĐIỆU, không có thì train vô nghĩa"

## 7. Sinh đặc trưng (spectrogram)

Mẫu thật và mẫu tổng hợp để **riêng hai thư mục**, không trộn chung.

Lý do: mẫu TTS sinh bằng giọng tiếng Anh nên **mất thanh huyền** (đo 06/08: whisper luôn đọc
"Vâng", không bao giờ "Vừng"). Chúng không dạy được đúng cụm; giá trị duy nhất là dạy biến thiên
giọng/tốc độ/kênh truyền. Tách ra thì đặt được tỉ trọng khác nhau ở bước sau — và tỉ trọng đó là
biến quan trọng nhất của cả quy trình.

In [ ]:
import os, shutil
os.chdir("/content/mww")
from mmap_ninja.ragged import RaggedMmap
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

# (thư mục nguồn, thư mục đích, số lần lặp lúc train)
# Mẫu thật ít (190) nên lặp nhiều hơn để bù số lượng; mẫu tổng hợp đã 1000 nên lặp ít.
TAP = [("real_samples", "features_real", 4),
       ("generated_samples", "features_synth", 2)]

augmenter = Augmentation(
    augmentation_duration_s=3.2,
    augmentation_probabilities={
        "SevenBandParametricEQ": 0.1, "TanhDistortion": 0.1, "PitchShift": 0.1,
        "BandStopFilter": 0.1, "AddColorNoise": 0.1, "AddBackgroundNoise": 0.75,
        "Gain": 1.0, "RIR": 0.5,
    },
    impulse_paths=["mit_rirs"],
    background_paths=["fma_16k", "audioset_16k"],
    background_min_snr_db=-5, background_max_snr_db=10,
    min_jitter_s=0.195, max_jitter_s=0.205,
)

for nguon, dich, lap_train in TAP:
    n_wav = len(os.listdir(nguon))
    print(f"\n=== {nguon} ({n_wav} wav) -> {dich}, lặp={lap_train} ===", flush=True)
    clips = Clips(input_directory=nguon, file_pattern="*.wav", max_clip_duration_s=None,
                  remove_silence=False, random_split_seed=10, split_count=0.1)
    os.makedirs(dich, exist_ok=True)
    for split, ten_split, lap, truot in (
        ("training", "train", lap_train, 10),
        ("validation", "validation", 1, 10),
        ("testing", "test", 1, 1),   # tập test chấm bằng bản streaming, không cần lặp khung
    ):
        out = f"{dich}/{split}"
        os.makedirs(out, exist_ok=True)
        # Kiểm thư mục CÓ DỮ LIỆU hay không, đừng chỉ kiểm nó tồn tại. Một lần chạy hỏng
        # giữa chừng để lại thư mục rỗng, và guard chỉ-kiểm-tồn-tại sẽ bỏ qua nó VĨNH VIỄN —
        # train vẫn chạy nhưng tập dương rỗng, ra accuracy 99,8% với recall 0%.
        mm = f"{out}/wakeword_mmap"
        if os.path.isdir(mm):
            try:
                n_co = len(RaggedMmap(mm))
            except Exception:
                n_co = 0
            if n_co > 0:
                print(f"  {split}: có rồi ({n_co} mẫu)"); continue
            print(f"  {split}: thư mục rỗng/hỏng -> xoá, sinh lại", flush=True)
            shutil.rmtree(mm, ignore_errors=True)
        spec = SpectrogramGeneration(clips=clips, augmenter=augmenter, slide_frames=truot,
                                     step_ms=10)
        RaggedMmap.from_generator(
            out_dir=f"{out}/wakeword_mmap",
            sample_generator=spec.spectrogram_generator(split=ten_split, repeat=lap),
            batch_size=100, verbose=True)
        print(f"  {split}: xong", flush=True)

# CHỨNG MINH có dữ liệu, đừng in "xong" rồi thôi. Tập dương rỗng là lỗi câm: train vẫn
# chạy, vẫn ra accuracy 99,8%, nhưng recall 0% vì model chỉ học "luôn trả lời không phải".
print("\n=== kiểm chứng ===")
tong = 0
for d in ("features_real", "features_synth"):
    for split in ("training", "validation", "testing"):
        n = len(RaggedMmap(f"{d}/{split}/wakeword_mmap"))
        tong += n
        print(f"  {'✅' if n else '❌'} {n:6d} mẫu  {d}/{split}")
assert tong > 0, "!! TẬP DƯƠNG RỖNG — kiểm lại real_samples/ và generated_samples/ ở ô 6"
print(f"\nSINH ĐẶC TRƯNG XONG — tổng {tong} mẫu dương")

## 7b. Đào hard negative tiếng Việt từ VIVOS

**Vì sao cần ô này:** vòng train đầu (07/08) KHÔNG có một mẫu hard negative nào — tập âm chỉ có
tiếng Anh và tiếng ồn chung chung. Kết quả đo trên board:

| Nói gì | dải `tb` | kích |
|---|---|---|
| "vừng ơi" | 0,991–0,996 | 16 |
| "vừng" một mình | tới 0,995 | 1 |
| **"ơi" một mình** | tới **0,996** | **7/~10** |

Ba phân bố trùng nhau ⇒ model kích trên bất kỳ âm tiết nào, không cần cả cặp. Không ngưỡng nào
cứu được. Nó học đúng thứ dữ liệu dạy nó; dữ liệu mới là thứ thiếu.

**Cách chữa:** VIVOS có 15 giờ, 50+ giọng Việt bản ngữ, **kèm transcript**. Lọc câu chứa từ đích
rồi dùng dấu thời gian TỪNG TỪ của faster-whisper để cắt đúng chỗ ⇒ ra hàng trăm mẫu "ơi" gần
như đứng một mình, từ nhiều giọng.

⚠️ Vì sao phải cắt theo từ chứ không lấy nguyên câu: một câu 4 giây chứa "ơi" trong 0,2 giây.
Đưa nguyên câu vào với `truncation_strategy: random` thì phần lớn cửa sổ 1,5 s **không chứa** từ
đích — tín hiệu bị pha loãng, và ta lại tốn một vòng train 1,5 giờ để phát hiện ra.

In [ ]:
# faster-whisper: đã là phụ thuộc sẵn của dự án (asr/), không phải công nghệ mới.
!pip install -q faster-whisper 2>&1 | tail -2

import os, re, unicodedata
from pathlib import Path
os.chdir("/content/mww")

# Từ đích và số mẫu mong muốn. Ưu tiên theo ĐO THẬT trên board, không theo cảm tính:
# "ơi" kích 7/10 nên cần nhiều nhất; "vừng" kích 1/10 nên ít hơn.
#
# 🔴 KẾT QUẢ THẬT 08/08 — VIVOS KHÔNG đáp ứng nổi chỉ tiêu này:
#     ơi 41/400 · vâng 5/200 · vùng 75/120 · vừa 120/120 · vừng 1/80  (tổng 242)
#     Chỉ 44 câu trên 11.660 câu chứa "ơi" = 0,38%.
#
# Vì sao: VIVOS là kho GIỌNG ĐỌC — người ta đọc câu soạn sẵn kiểu văn viết. "ơi" và "vâng" là
# từ đệm chỉ sống trong HỘI THOẠI, gần như không xuất hiện trong văn bản để đọc. Chọn kho theo
# tiêu chí "có transcript" là chưa đủ; phải hỏi thêm transcript của LOẠI LỜI NÓI nào.
#
# Muốn đủ mẫu "ơi" thì cần nguồn hội thoại thật (podcast/vlog/phỏng vấn). Nguồn đó không có
# transcript, nhưng KHÔNG sao: bỏ bước lọc chữ, cho whisper chạy thẳng trên audio rồi lọc theo
# `w.word` ở ô kế tiếp — chậm hơn nhiều nhưng cùng một đường code.
TU_DICH = {
    "ơi":   400,   # thủ phạm nặng nhất
    "vâng": 200,   # cực phổ biến trong tiếng Việt, gần âm "vừng"
    "vùng": 120,
    "vừa":  120,
    "vừng":  80,   # hiếm (hạt vừng) — nhiều khả năng không đủ, chấp nhận
}

def bo_dau_cau(s: str) -> str:
    return re.sub(r"[^\w\sÀ-ỹ]", " ", s.lower())

# trust_remote_code=True: VIVOS nạp bằng script của chính repo AILAB-VNUHCM. Thiếu cờ này thì
# datasets DỪNG LẠI hỏi [y/N] giữa chừng — ô treo im lặng chứ không báo lỗi.
from datasets import load_dataset
print("tải VIVOS (15 giờ, 50+ giọng)...", flush=True)
try:
    vivos = load_dataset("AILAB-VNUHCM/vivos", split="train", trust_remote_code=True)
except Exception as e:
    print("AILAB-VNUHCM/vivos lỗi:", type(e).__name__, "-> thử 'vivos'", flush=True)
    vivos = load_dataset("vivos", split="train", trust_remote_code=True)

print(f"VIVOS: {len(vivos)} câu")
cot_text = "sentence" if "sentence" in vivos.column_names else "transcription"
print("cột transcript:", cot_text)

# Lọc trước bằng transcript — rẻ. Chỉ những câu lọt mới phải chạy whisper (đắt).
ung_vien = {t: [] for t in TU_DICH}
for i, row in enumerate(vivos):
    tu = bo_dau_cau(row[cot_text]).split()
    for t in TU_DICH:
        if t in tu and len(ung_vien[t]) < TU_DICH[t] * 2:   # lấy dư 2x, whisper sẽ loại bớt
            ung_vien[t].append(i)

print("\n=== câu ứng viên tìm được ===")
for t, ds in ung_vien.items():
    print(f"  {t:6s}: {len(ds):5d} câu  (mục tiêu {TU_DICH[t]} mẫu)")

In [ ]:
# Cắt từ đích ra khỏi câu bằng dấu thời gian TỪNG TỪ.
import numpy as np, soundfile as sf, librosa
from faster_whisper import WhisperModel

SR = 16000
CUA_SO_S = 1.5          # đúng clip_duration_ms của model
OUT = Path("/content/mww/hard_negatives"); OUT.mkdir(exist_ok=True)

model = WhisperModel("small", device="cuda", compute_type="float16")
print("faster-whisper sẵn sàng\n", flush=True)

def chuan(s):
    return re.sub(r"[^\w\sÀ-ỹ]", "", s.lower()).strip()

tong = {}
for tu, chi_so in ung_vien.items():
    n = 0
    for idx in chi_so:
        if n >= TU_DICH[tu]:
            break
        row = vivos[idx]
        y = row["audio"]["array"].astype(np.float32)
        sr_goc = row["audio"]["sampling_rate"]
        if sr_goc != SR:
            y = librosa.resample(y, orig_sr=sr_goc, target_sr=SR)

        try:
            segs, _ = model.transcribe(y, language="vi", word_timestamps=True,
                                       condition_on_previous_text=False)
            words = [w for s in segs for w in (s.words or [])]
        except Exception:
            continue

        for w in words:
            if chuan(w.word) != tu:
                continue
            # Cửa sổ 1,5s ĐẶT GIỮA từ đích -> từ nằm giữa, hai bên là ngữ cảnh thật.
            giua = (w.start + w.end) / 2.0
            a = max(0.0, giua - CUA_SO_S / 2)
            lat = y[int(a * SR): int((a + CUA_SO_S) * SR)]
            if len(lat) < int(0.8 * SR):     # quá ngắn (từ ở sát mép câu) -> bỏ
                continue
            if len(lat) < int(CUA_SO_S * SR):
                lat = np.pad(lat, (0, int(CUA_SO_S * SR) - len(lat)))
            sf.write(OUT / f"{tu}-{n:04d}.wav", (lat * 32767).astype(np.int16), SR,
                     subtype="PCM_16")
            n += 1
            break        # mỗi câu lấy 1 mẫu, giữ đa dạng giọng thay vì lặp một người
    tong[tu] = n
    print(f"  {tu:6s}: {n:4d} mẫu", flush=True)

print(f"\nTỔNG hard negative: {sum(tong.values())} mẫu -> {OUT}")
assert sum(tong.values()) > 100, "!! quá ít mẫu — kiểm lại bước lọc transcript"

### Sinh đặc trưng cho hard negative

Dùng chính bộ augment của mẫu dương để hai bên cùng điều kiện kênh truyền — nếu mẫu âm sạch hơn
mẫu dương thì model học phân biệt **độ sạch** thay vì học nội dung.

`truth=False` là khác biệt duy nhất so với ô mẫu dương.

In [ ]:
import os, shutil
os.chdir("/content/mww")
from pathlib import Path
from mmap_ninja.ragged import RaggedMmap
from microwakeword.audio.clips import Clips
from microwakeword.audio.spectrograms import SpectrogramGeneration

# `augmenter` do ô "Sinh đặc trưng" phía trên tạo ra — dùng LẠI đúng bộ đó, không dựng bộ mới.
# Nếu mẫu âm sạch hơn mẫu dương thì model học phân biệt ĐỘ SẠCH thay vì học nội dung.
assert "augmenter" in dir(), "!! chạy ô 'Sinh đặc trưng' trước ô này"

clips_hn = Clips(input_directory="hard_negatives", file_pattern="*.wav",
                 max_clip_duration_s=None, remove_silence=False,
                 random_split_seed=10, split_count=0.1)

Path("features_hard_neg").mkdir(exist_ok=True)
for split, ten, lap, truot in (("training","train",2,10), ("validation","validation",1,10),
                               ("testing","test",1,1)):
    out = Path("features_hard_neg") / split
    out.mkdir(exist_ok=True)
    mm = out / "wakeword_mmap"
    if mm.is_dir():
        try:    n_co = len(RaggedMmap(str(mm)))
        except Exception: n_co = 0
        if n_co > 0:
            print(f"  {split}: có rồi ({n_co} mẫu)"); continue
        shutil.rmtree(mm, ignore_errors=True)
    spec = SpectrogramGeneration(clips=clips_hn, augmenter=augmenter,
                                 slide_frames=truot, step_ms=10)
    RaggedMmap.from_generator(out_dir=str(mm),
        sample_generator=spec.spectrogram_generator(split=ten, repeat=lap),
        batch_size=100, verbose=True)
    print(f"  {split}: xong", flush=True)

tong_hn = sum(len(RaggedMmap(f"features_hard_neg/{s}/wakeword_mmap"))
              for s in ("training","validation","testing"))
print(f"\nĐẶC TRƯNG HARD NEGATIVE: {tong_hn} mẫu")
assert tong_hn > 0, "!! tập hard negative RỖNG"

## 8. Viết cấu hình train

In [ ]:
import yaml, os
W = "/content/mww"

cfg = {
    "window_step_ms": 10,
    "train_dir": f"{W}/trained_models/vung_oi",
    "features": [
        # --- DƯƠNG ---
        # Mẫu thật: ÍT nhưng là thứ DUY NHẤT mang thanh điệu. Trọng số cao nhất.
        {"features_dir": f"{W}/features_real", "sampling_weight": 4.0, "penalty_weight": 1.5,
         "truth": True, "truncation_strategy": "truncate_start", "type": "mmap"},
        # Mẫu TTS: nhiều nhưng sai thanh điệu. Bỏ hẳn thì mất phần dạy biến thiên giọng;
        # để cao thì model học nhầm cụm phẳng ("vâng ơi").
        {"features_dir": f"{W}/features_synth", "sampling_weight": 1.0, "penalty_weight": 1.0,
         "truth": True, "truncation_strategy": "truncate_start", "type": "mmap"},
        # --- ÂM ---
        # Hard negative đào từ VIVOS: trọng số CAO và penalty CAO nhất trong mọi tập.
        #
        # Vì sao penalty 3.0 — cao hơn cả mẫu dương thật (1.5): đây chính là lỗi vòng trước.
        # Model kích 7/10 lần khi nghe "ơi" một mình vì nó CHƯA TỪNG bị phạt vì việc đó. Đặt
        # penalty cao là nói với model rằng nhầm ở đây đắt hơn mọi loại nhầm khác.
        #
        # truncate_start chứ không random: clip đã được cắt 1,5s ĐẶT GIỮA từ đích rồi, cắt ngẫu
        # nhiên nữa sẽ đẩy từ đích ra khỏi cửa sổ và làm hỏng chính thứ ta muốn dạy.
        {"features_dir": f"{W}/features_hard_neg", "sampling_weight": 8.0,
         "penalty_weight": 3.0, "truth": False, "truncation_strategy": "truncate_start",
         "type": "mmap"},
        {"features_dir": f"{W}/negative_datasets/speech", "sampling_weight": 10.0,
         "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
        {"features_dir": f"{W}/negative_datasets/dinner_party", "sampling_weight": 10.0,
         "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
        {"features_dir": f"{W}/negative_datasets/no_speech", "sampling_weight": 5.0,
         "penalty_weight": 1.0, "truth": False, "truncation_strategy": "random", "type": "mmap"},
        # weight 0: chỉ dùng để đo báo giả/giờ, không tham gia train.
        {"features_dir": f"{W}/negative_datasets/dinner_party_eval", "sampling_weight": 0.0,
         "penalty_weight": 1.0, "truth": False, "truncation_strategy": "split", "type": "mmap"},
    ],
    # 128 là giá trị tham chiếu của tác giả. Dưới WSL phải hạ xuống 64 vì thiếu RAM — Colab
    # không có ràng buộc đó nên dùng lại số gốc.
    "batch_size": 128,
    # 20:1 nghiêng về phía ÂM — đây là thứ giữ báo giả thấp. Bỏ khoá này thì mặc định 1:1 và
    # model bắt tốt nhưng kích bừa.
    "positive_class_weight": [1],
    "negative_class_weight": [20],
    "training_steps": [5000, 5000, 5000],
    "learning_rates": [0.001, 0.0005, 0.0001],
    "mix_up_augmentation_prob": [0.0, 0.0, 0.0],
    "freq_mask_max_size": [0, 0, 0], "freq_mask_count": [0, 0, 0],
    "time_mask_max_size": [0, 0, 0], "time_mask_count": [0, 0, 0],
    "eval_step_interval": 500,
    "clip_duration_ms": 1500,
    # Chọn checkpoint theo tỉ lệ bắt trung bình tại các mức báo giả chấp nhận được (≤2/giờ),
    # KHÔNG phải theo accuracy — accuracy vô dụng khi lớp âm áp đảo.
    "target_minimization": 0.9,
    "minimization_metric": None,
    "maximization_metric": "average_viable_recall",
}

os.makedirs(f"{W}/trained_models", exist_ok=True)
with open(f"{W}/training_parameters.yaml", "w") as f:
    yaml.dump(cfg, f, sort_keys=False, allow_unicode=True)
print("đã ghi training_parameters.yaml")

## 9. Train

⚠️ **`--stride 3` phải giữ nguyên.** Nó quyết định `KWS_STEPS_PER_INVOKE=3` bên firmware.
Đổi stride là phải sửa hằng số bên kia — cái bẫy này đã dính một lần rồi.

Trên T4 khoảng 20-40 phút. Mốc đáng tin là dòng `Step #500 (nonstreaming): Validation: ...` —
đó là chỗ hai lần train dưới WSL đều gục.

In [ ]:
import shutil
# microwakeword TỪ CHỐI ghi đè thư mục kết quả đã có ("model already exists in folder").
# Một lần chạy hỏng/bị ngắt cũng đủ để lại thư mục đó, và mọi lần chạy sau đều chết ngay
# ở dòng đầu. Dọn trước cho khỏi vấp — `--restore_checkpoint 0` vốn đã train lại từ đầu.
shutil.rmtree("/content/mww/trained_models/vung_oi", ignore_errors=True)
print("đã dọn thư mục kết quả cũ\n")

%cd /content/mww
!python -m microwakeword.model_train_eval \
    --training_config=/content/mww/training_parameters.yaml \
    --train 1 \
    --restore_checkpoint 0 \
    --test_tf_nonstreaming 0 \
    --test_tflite_nonstreaming 0 \
    --test_tflite_nonstreaming_quantized 0 \
    --test_tflite_streaming 0 \
    --test_tflite_streaming_quantized 1 \
    --use_weights best_weights \
    mixednet \
    --pointwise_filters "64,64,64,64" \
    --repeat_in_block "1,1,1,1" \
    --mixconv_kernel_sizes "[5],[7,11],[9,15],[23]" \
    --residual_connection "0,0,0,0" \
    --first_conv_filters 32 \
    --first_conv_kernel_size 5 \
    --stride 3

## 10. Kiểm tập op ⊆ 13 op firmware đang đăng ký

Firmware đăng ký đúng 13 op trong `kws.cpp`. Model dùng op ngoài danh sách đó thì
`AllocateTensors()` **chết trên board** — phải bắt ở đây, không phải sau khi nạp.

In [ ]:
import struct, glob

NAMES = """ADD AVERAGE_POOL_2D CONCATENATION CONV_2D DEPTHWISE_CONV_2D DEPTH_TO_SPACE DEQUANTIZE
EMBEDDING_LOOKUP FLOOR FULLY_CONNECTED HASHTABLE_LOOKUP L2_NORMALIZATION L2_POOL_2D
LOCAL_RESPONSE_NORMALIZATION LOGISTIC LSH_PROJECTION LSTM MAX_POOL_2D MUL RELU RELU_N1_TO_1 RELU6
RESHAPE RESIZE_BILINEAR RNN SOFTMAX SPACE_TO_DEPTH SVDF TANH CONCAT_EMBEDDINGS SKIP_GRAM CALL
CUSTOM EMBEDDING_LOOKUP_SPARSE PAD UNIDIRECTIONAL_SEQUENCE_RNN GATHER BATCH_TO_SPACE_ND
SPACE_TO_BATCH_ND TRANSPOSE MEAN SUB DIV SQUEEZE UNIDIRECTIONAL_SEQUENCE_LSTM STRIDED_SLICE
BIDIRECTIONAL_SEQUENCE_RNN EXP TOPK_V2 SPLIT LOG_SOFTMAX DELEGATE BIDIRECTIONAL_SEQUENCE_LSTM
CAST PRELU MAXIMUM ARG_MAX MINIMUM LESS NEG PADV2 GREATER GREATER_EQUAL LESS_EQUAL SELECT SLICE
SIN TRANSPOSE_CONV SPARSE_TO_DENSE TILE EXPAND_DIMS EQUAL NOT_EQUAL LOG SUM SQRT RSQRT SHAPE POW
ARG_MIN FAKE_QUANT REDUCE_PROD REDUCE_MAX PACK LOGICAL_OR ONE_HOT LOGICAL_AND LOGICAL_NOT UNPACK
REDUCE_MIN FLOOR_DIV REDUCE_ANY SQUARE ZEROS_LIKE FILL FLOOR_MOD RANGE RESIZE_NEAREST_NEIGHBOR
LEAKY_RELU SQUARED_DIFFERENCE MIRROR_PAD ABS SPLIT_V UNIQUE CEIL REVERSE_V2 ADD_N GATHER_ND COS
WHERE RANK ELU REVERSE_SEQUENCE MATRIX_DIAG QUANTIZE MATRIX_SET_DIAG ROUND HARD_SWISH IF WHILE
NON_MAX_SUPPRESSION_V4 NON_MAX_SUPPRESSION_V5 SCATTER_ND SELECT_V2 DENSIFY SEGMENT_SUM BATCH_MATMUL
PLACEHOLDER_FOR_GREATER_OP_CODES CUMSUM CALL_ONCE BROADCAST_TO RFFT2D CONV_3D IMAG REAL COMPLEX_ABS
HASHTABLE HASHTABLE_FIND HASHTABLE_IMPORT HASHTABLE_SIZE REDUCE_ALL CONV_3D_TRANSPOSE VAR_HANDLE
READ_VARIABLE ASSIGN_VARIABLE BROADCAST_ARGS RANDOM_STANDARD_NORMAL BUCKETIZE RANDOM_UNIFORM
MULTINOMIAL GELU DYNAMIC_UPDATE_SLICE RELU_0_TO_1""".split()


def _u32(b, o): return struct.unpack_from("<I", b, o)[0]
def _i32(b, o): return struct.unpack_from("<i", b, o)[0]


def _field(b, tbl, idx):
    vt = tbl - _i32(b, tbl)
    vlen = struct.unpack_from("<H", b, vt)[0]
    pos = 4 + idx * 2
    if pos >= vlen:
        return None
    off = struct.unpack_from("<H", b, vt + pos)[0]
    return tbl + off if off else None


def doc_op(path):
    """Đọc Model.operator_codes thẳng từ flatbuffer — không cần schema hay lib phụ."""
    b = open(path, "rb").read()
    f = _field(b, _u32(b, 0), 1)
    if f is None:
        return []
    vec = f + _u32(b, f)
    out = []
    for k in range(_u32(b, vec)):
        e = vec + 4 + k * 4
        oc = e + _u32(b, e)
        dep, bi = _field(b, oc, 0), _field(b, oc, 3)
        code = _i32(b, bi) if bi is not None else (b[dep] if dep is not None else 0)
        out.append(NAMES[code] if code < len(NAMES) else f"<{code}>")
    return out


FIRMWARE_CO = set("""ASSIGN_VARIABLE CALL_ONCE CONCATENATION CONV_2D DEPTHWISE_CONV_2D
FULLY_CONNECTED LOGISTIC QUANTIZE READ_VARIABLE RESHAPE SPLIT_V STRIDED_SLICE VAR_HANDLE""".split())

tfl = glob.glob("/content/mww/trained_models/vung_oi/**/*.tflite", recursive=True)
assert tfl, "!! chưa có .tflite — ô train chưa chạy xong hoặc đã lỗi"
for p in tfl:
    can = set(doc_op(p))
    thieu = can - FIRMWARE_CO
    print(f"\n{p}")
    print("  op dùng:", ", ".join(sorted(can)))
    if thieu:
        print("  !! THIẾU trong resolver:", ", ".join(sorted(thieu)))
        print("     Phải thêm vào kNumOps + resolver trong kws.cpp TRƯỚC khi build.")
    else:
        print("  OK: mọi op đều đã đăng ký ở firmware")

## 11. Tải kết quả về

Tải file `vung-oi-ket-qua.tar.gz` về máy. Bên trong có `.tflite` streaming đã lượng tử hoá
cùng các số đo để chọn ngưỡng.

⚠️ **Đừng lấy `tensor_arena_size` từ manifest.** Đã xác nhận trên 2 model rằng manifest luôn
ghi thấp hơn thực tế (`okay_nabu` ghi 26 080 / thật 33 916; `alexa` ghi 22 348 / thật 30 256) vì
nó không tính buffer `MicroResourceVariables`. Đọc `arena_dung` mà firmware in ra lúc init.

In [ ]:
import os, glob, tarfile
os.chdir("/content/mww/trained_models")

with tarfile.open("/content/vung-oi-ket-qua.tar.gz", "w:gz") as t:
    t.add("vung_oi")
print("gói xong:", os.path.getsize("/content/vung-oi-ket-qua.tar.gz") / 1e6, "MB")

print("\n=== file trong gói ===")
for p in sorted(glob.glob("vung_oi/**/*", recursive=True)):
    if os.path.isfile(p):
        print(f"  {os.path.getsize(p)/1024:8.1f} KB  {p}")

from google.colab import files
files.download("/content/vung-oi-ket-qua.tar.gz")